# # Merge FiveThirtyEight Elo ratings into `full_nba`
This notebook

1. reads  
   * **data/processed/full_nba.csv**  (whatever you built in *merge_data.ipynb*)  
   * **data/raw/nbaallelo.csv**       (FiveThirtyEight’s Elo ratings)

2. adds the columns we care about from the Elo table (`elo_i`, `elo_n`, `win_equiv`, `forecast`, …)

3. saves the combined table to **data/processed/full_nba_with_elo.csv**

In [1]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------------
# Folder layout
# ./data
# ├── raw/        … nbaallelo.csv lives here
# └── processed/  … full_nba.csv already lives here
# ------------------------------------------------------------------
RAW_DIR       = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

# Read both tables --------------------------------------------------
full_nba = pd.read_csv(PROCESSED_DIR / "full_nba.csv")
elo      = pd.read_csv(RAW_DIR / "nbaallelo.csv")
print("full_nba →", full_nba.shape, "rows × cols")
print("elo      →", elo.shape,      "rows × cols")


full_nba → (116298, 91) rows × cols
elo      → (126314, 23) rows × cols


In [2]:
# ------------------------------------------------------------
# 2.  Ensure `date_game` is datetime in *both* frames
# ------------------------------------------------------------
full_nba["date_game"] = pd.to_datetime(full_nba["date_game"]).dt.date
elo["date_game"]      = pd.to_datetime(elo["date_game"]).dt.date

In [3]:
# ───────────────────────────────────────────────────────────────
# 3  Slice / rename only the Elo columns you want to keep
# ───────────────────────────────────────────────────────────────
elo_keep = (
    elo[[
        "date_game",             # join key
        "team_id",               # join key (main)
        "opp_id",                # join key (opp)
        "is_playoffs",
        "elo_i",   "elo_n",
        "win_equiv",
        "opp_elo_i", "opp_elo_n",
        "game_result",
        "forecast",
    ]]
    .rename(columns={
        "elo_i":      "elo_main_pre_538",
        "elo_n":      "elo_main_post_538",
        "opp_elo_i":  "elo_opp_pre_538",
        "opp_elo_n":  "elo_opp_post_538",
    })
)

In [4]:
# ───────────────────────────────────────────────────────────────
# 4  INNER-join on (date_game, team_abbreviation_main ↔ team_id,
#                   team_abbreviation_opp ↔ opp_id)
# ───────────────────────────────────────────────────────────────
merged = (
    full_nba
        .merge(
            elo_keep,
            how="inner",
            left_on = ["date_game",
                       "team_abbreviation_main",
                       "team_abbreviation_opp"],
            right_on= ["date_game",
                       "team_id",
                       "opp_id"],
            validate="m:m",        # many↔many allowed
        )
        .drop(columns=["team_id", "opp_id"])  # throw away Elo’s join cols
)

print("Rows kept after inner join:", len(merged))

Rows kept after inner join: 69014


In [5]:
print(merged.columns)

Index(['game_id', 'date_game', 'team_id_main', 'team_abbreviation_main',
       'team_city_name_main', 'team_nickname_main', 'team_wins_losses_main',
       'pts_qtr1_main', 'pts_qtr2_main', 'pts_qtr3_main', 'pts_qtr4_main',
       'pts_ot1_main', 'pts_ot2_main', 'pts_ot3_main', 'pts_ot4_main',
       'pts_ot5_main', 'pts_ot6_main', 'pts_ot7_main', 'pts_ot8_main',
       'pts_ot9_main', 'pts_ot10_main', 'pts_main', 'team_name_main',
       'matchup_main', 'wl_main', 'fgm_main', 'fga_main', 'fg_pct_main',
       'fg3m_main', 'fg3a_main', 'fg3_pct_main', 'ftm_main', 'fta_main',
       'ft_pct_main', 'oreb_main', 'dreb_main', 'reb_main', 'ast_main',
       'stl_main', 'blk_main', 'tov_main', 'pf_main', 'plus_minus_main',
       'video_available_main', 'team_id_opp', 'team_abbreviation_opp',
       'team_city_name_opp', 'team_nickname_opp', 'team_wins_losses_opp',
       'pts_qtr1_opp', 'pts_qtr2_opp', 'pts_qtr3_opp', 'pts_qtr4_opp',
       'pts_ot1_opp', 'pts_ot2_opp', 'pts_ot3_opp', 'pts

In [7]:
# ───────────────────────────────────────────────────────────────
# 5  (Optional) save the result
# ───────────────────────────────────────────────────────────────
out_path = PROCESSED_DIR / "full_nba_with_elo_538.csv"
merged.to_csv(out_path, index=False)
print(f"✓ Saved → {out_path}") 

✓ Saved → ..\data\processed\full_nba_with_elo_538.csv
